# CS 297 - Experiment 1
### Description: Can a small, open weight model reliably detect noise in an image and determine whether denoising is required?
### Ethan Machado


## Experiment design details:

Classification problems:

1.   Noise evaluation task - binary class problem (0 = clean; 1 = augmented)
2.   Noise severity task - multi-class problem (0 = clean; 1 = weak; 2 = medium; 3 = strong)
3.   Routing decision task - (clean, weak --> no denoising; medium, strong --> yes denoising)

Pipeline:

1.   Supply a small resized overview plus a native-resolution 512×512 center detail crop. Apply identical preparation to references and targets; source files remain unchanged.
2.   Evaluate the model's performance using zero-shot prompting.
3.   Evaluate the model's performance using few-shot prompting.

Evaluation criteria:

For the noise evaluation task: Accuracy, Precision, Recall, Macro-F1, Confusion matrix.

For the noise severity task: MAE loss function, Accuracy, Precision, Recall, Macro-F1, Confusion matrix.

For the routing decision task: Routing accuracy, Precision, Recall, Macro-F1, Confusion matrix.

**Note: There should be an additional test performed to check the stability of decision-making by processing images multiple times during each experiment.**










In [1]:
# Local setup: run with the project's .venv notebook kernel.
import os
import sys
from pathlib import Path

PROJECT_PATH = '/Users/ethanmachado/Library/CloudStorage/GoogleDrive-ethan.machado@sjsu.edu/My Drive/04_F2026/CS297/experiment1'
os.chdir(PROJECT_PATH)
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)
os.environ["HF_HOME"] = os.path.join(PROJECT_PATH, "hf_cache")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
print("Project:", PROJECT_PATH)


Project: /Users/ethanmachado/Library/CloudStorage/GoogleDrive-ethan.machado@sjsu.edu/My Drive/04_F2026/CS297/experiment1


In [2]:
# Install requirements.txt in the project's .venv before running this notebook.
import torch
import pandas as pd
from IPython.display import display
from transformers import AutoProcessor, AutoModelForMultimodalLM

# Run All defaults to a small end-to-end check: 4 images x 3 passes.
SMALL_TEST = False  # start with four evaluation images
TEST_SUBJECTS = 1  # each subject has clean/weak/medium/strong versions
BATCH_SIZE = 1  # more visual tokens per target, especially in few-shot mode
FULL_RESOLUTION = False  # resized overview plus native-resolution detail crop
MAX_PIXELS = 262144  # small overview budget; preserves aspect ratio
DETAIL_CROP_SIZE = 512  # native center crop preserves local grain (None = overview only)
INFERENCE_MODE = "zero_shot"  # switch to "few_shot" and rerun from the split cell
EXPERIMENT_NUMBER = 4  # small overview + native 512x512 detail crop
STABILITY_PASSES = 3
EXAMPLE_SUBJECT = "0"
MODEL_NAME = "Qwen/Qwen3.5-0.8B"
OUTPUT_DIR = os.path.join(PROJECT_PATH, "outputs", "notebook_small_test" if SMALL_TEST else "notebook_experiments")


W1020 11:11:33.847000 22472 .venv/lib/python3.14/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [3]:
from src.image_quality_pipeline import build_dataset

df_experiment = build_dataset(os.path.join(PROJECT_PATH, "data", "files"))
print(f"Available images: {len(df_experiment)}")
display(df_experiment.groupby("severity").size().rename("images"))


Available images: 200


severity
0    50
1    50
2    50
3    50
Name: images, dtype: int64

In [4]:
# Local cached-model inference needs no OpenAI client or API token.
# The cached processor/model must already exist under hf_cache.
print("Using cached model:", MODEL_NAME)


Using cached model: Qwen/Qwen3.5-0.8B


In [5]:
print("Loading cached processor")
processor = AutoProcessor.from_pretrained(MODEL_NAME, local_files_only=True)
if MAX_PIXELS is not None:
    if MAX_PIXELS < processor.image_processor.size["shortest_edge"]:
        raise ValueError("MAX_PIXELS must be at least the processor minimum")
    processor.image_processor.size["longest_edge"] = MAX_PIXELS

print("Loading cached model")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = AutoModelForMultimodalLM.from_pretrained(MODEL_NAME, local_files_only=True).to(device)
print("Device:", model.device)
print("Processor budget (ignored in full-resolution mode):", dict(processor.image_processor.size))


Loading cached processor
Loading cached model


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Device: mps:0
Processor budget (ignored in full-resolution mode): {'longest_edge': 262144, 'shortest_edge': 65536}


In [6]:
from src.image_quality_pipeline import prepare_few_shot_split, task_view

# Both modes reserve the same example subject, including in a small test.
df_evaluation, few_shot_examples = prepare_few_shot_split(df_experiment, EXAMPLE_SUBJECT)
if SMALL_TEST:
    if isinstance(TEST_SUBJECTS, bool) or not isinstance(TEST_SUBJECTS, int) or TEST_SUBJECTS < 1:
        raise ValueError("TEST_SUBJECTS must be a positive integer")
    selected_subjects = sorted(df_evaluation["subject_id"].unique())[:TEST_SUBJECTS]
    df_evaluation = df_evaluation.loc[df_evaluation["subject_id"].isin(selected_subjects)].copy()

noise_eval = task_view(df_evaluation, "noise")
severity_eval = task_view(df_evaluation, "severity")
route_eval = task_view(df_evaluation, "route")
print(f"Mode: {INFERENCE_MODE}; reference subject: {EXAMPLE_SUBJECT}")
print(f"Evaluation images: {len(df_evaluation)}; passes: {STABILITY_PASSES}")
print(f"Batches per pass: {(len(df_evaluation) + BATCH_SIZE - 1) // BATCH_SIZE}")
print("Output directory:", OUTPUT_DIR)
display(pd.DataFrame([{"image_path": example.image_path, **example.prediction}
                      for example in few_shot_examples]))


Mode: zero_shot; reference subject: 0
Evaluation images: 196; passes: 3
Batches per pass: 196
Output directory: /Users/ethanmachado/Library/CloudStorage/GoogleDrive-ethan.machado@sjsu.edu/My Drive/04_F2026/CS297/experiment1/outputs/notebook_experiments


,image_path,noise,severity,route
0,/Users/ethanmachado/Library/CloudStorage/Googl...,0,0,0
1,/Users/ethanmachado/Library/CloudStorage/Googl...,1,1,0
2,/Users/ethanmachado/Library/CloudStorage/Googl...,1,2,1
3,/Users/ethanmachado/Library/CloudStorage/Googl...,1,3,1


In [7]:
# Keep model-specific batched generation in the supporting Python script.
from src.transformers_adapter import TransformersImageModel

model_adapter = TransformersImageModel(
    model, processor, max_new_tokens=128,
    detail_crop_size=None if FULL_RESOLUTION else DETAIL_CROP_SIZE,
    full_resolution=FULL_RESOLUTION,
)
print("Image preparation:", model_adapter.inference_config)


Image preparation: {'max_new_tokens': 128, 'do_sample': False, 'dtype': 'torch.bfloat16', 'device': 'mps:0', 'model_source': 'Qwen/Qwen3.5-0.8B', 'processor_size': {'longest_edge': 262144, 'shortest_edge': 65536}, 'detail_crop_size': 512, 'image_views': 'overview_and_center_detail', 'full_resolution': False, 'padding': None, 'processor_do_resize': True, 'exif_transpose': True, 'model_revision': None}


### Image preparation and calibration
The recommended configuration uses `FULL_RESOLUTION = False`, `MAX_PIXELS = 262144`, `DETAIL_CROP_SIZE = 512`, and `BATCH_SIZE = 1`. Each image supplies an aspect-ratio-preserving overview plus a 512×512 center crop taken directly from the source pixels. The crop preserves local grain that overview downsampling can suppress. A center crop may miss localized degradation elsewhere. References and targets use identical preparation.

Experiment 4 identifies this configuration, separately from the larger overview/detail configuration (experiment 2) and full-resolution configuration (experiment 3). `SMALL_TEST = True` selects four evaluation images; set it to `False` for the full split. `STABILITY_PASSES = 3` repeats the selected images three times. The stability runner requires at least two passes.

Restart the kernel and Run All to apply the settings and updated shared modules. To compare zero-shot and few-shot, switch `INFERENCE_MODE` and rerun from the split cell. Changing `MAX_PIXELS` requires processor/adapter recreation; changing crop or full-resolution settings requires adapter recreation. Existing results remain in their original directories.


In [8]:
from src.evaluation import EvaluationConfig, run_stability_experiment

# Each pass is checkpointed as a normal run; the group report links all passes.
# Run this cell once per INFERENCE_MODE to compare zero-shot and few-shot.
evaluation_config = EvaluationConfig(
    output_dir=OUTPUT_DIR,
    experiment_number=EXPERIMENT_NUMBER,
    stability_passes=STABILITY_PASSES,
)
experiment_report = run_stability_experiment(
    model=model_adapter,
    image_paths=df_evaluation["file_path"].tolist(),
    labels=df_evaluation,
    config=evaluation_config,
    batch_size=BATCH_SIZE,
    inference_mode=INFERENCE_MODE,
    examples=few_shot_examples,
    model_name=MODEL_NAME,
    show_progress=True,
)

# Keep each pass's task metrics separate from the stability measurements.
metrics = experiment_report["metrics"][0]
metric_rows = [
    {"pass": pass_number, "task": task,
     **{key: values.get(key) for key in
        ("accuracy", "precision", "recall", "macro_f1", "mae", "coverage")}}
    for pass_number, pass_metrics in enumerate(experiment_report["metrics"], 1)
    for task, values in pass_metrics.items()
]
display(pd.DataFrame(metric_rows))
for pass_number, pass_metrics in enumerate(experiment_report["metrics"], 1):
    for task, values in pass_metrics.items():
        print(f"Pass {pass_number}: {task} confusion matrix (true rows, predicted columns)")
        display(pd.DataFrame(values["confusion_matrix"], index=values["labels"], columns=values["labels"]))

display(pd.DataFrame({
    task: {key: value for key, value in values.items() if key != "per_image"}
    for task, values in experiment_report["stability"]["tasks"].items()
}).T)
print("Saved report:", experiment_report["report_path"])


Inference:   0%|          | 0/196 [00:00<?, ?image/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Inference:   0%|          | 0/196 [00:00<?, ?image/s]

Inference:   0%|          | 0/196 [00:00<?, ?image/s]

,pass,task,accuracy,precision,recall,macro_f1,mae,coverage
0,1,noise,0.26,0.260000,1.000000,0.206349,NaN,0.255102
1,1,severity,0.14,0.107724,0.354167,0.157938,0.88,0.255102
2,1,route,0.92,0.666667,0.857143,0.851190,NaN,0.255102
3,2,noise,0.26,0.260000,1.000000,0.206349,NaN,0.255102
4,2,severity,0.14,0.107724,0.354167,0.157938,0.88,0.255102
5,2,route,0.92,0.666667,0.857143,0.851190,NaN,0.255102
6,3,noise,0.26,0.260000,1.000000,0.206349,NaN,0.255102
7,3,severity,0.14,0.107724,0.354167,0.157938,0.88,0.255102
8,3,route,0.92,0.666667,0.857143,0.851190,NaN,0.255102


Pass 1: noise confusion matrix (true rows, predicted columns)


... (15 outputs left out)

Pass 3: route confusion matrix (true rows, predicted columns)


,0,1
0,40,3
1,1,6


,complete_images,incomplete_images,stable_images,changed_images,coverage,stable_fraction,stable_fraction_complete,valid_pairs,agreeing_pairs,pairwise_agreement
noise,50.0,146.0,50.0,0.0,0.255102,0.255102,1.0,150.0,150.0,1.0
severity,50.0,146.0,50.0,0.0,0.255102,0.255102,1.0,150.0,150.0,1.0
route,50.0,146.0,50.0,0.0,0.255102,0.255102,1.0,150.0,150.0,1.0
joint,50.0,146.0,50.0,0.0,0.255102,0.255102,1.0,150.0,150.0,1.0


Saved report: /Users/ethanmachado/Library/CloudStorage/GoogleDrive-ethan.machado@sjsu.edu/My Drive/04_F2026/CS297/experiment1/outputs/notebook_experiments/experiment_004/stability_001/report.json


## Evaluate previously saved runs
This cell needs no model or image loading. Set the run paths to existing saved runs.

In [9]:
from pathlib import Path
from src.evaluation import evaluate_saved_run, evaluate_stability

# Set these to runs from the same model, prompt, split, and experiment.
# Prefer the exact group produced above; edit the fallback for an offline session.
SAVED_RUNS = ([Path(path) for path in experiment_report["run_dirs"]]
              if "experiment_report" in globals() else [
                  Path(OUTPUT_DIR) / f"experiment_{EXPERIMENT_NUMBER:03d}" / f"run_{number:03d}"
                  for number in (1, 2, 3)
              ])
# Rewrites metrics.json with the full metrics for each selected run.
if all((run / "labels.json").exists() for run in SAVED_RUNS):
    saved_metrics = [evaluate_saved_run(run) for run in SAVED_RUNS]
    saved_stability = evaluate_stability(SAVED_RUNS)
    display(pd.DataFrame({task: {k: v for k, v in values.items() if k != "per_image"}
                          for task, values in saved_stability["tasks"].items()}).T)
else:
    print("Set SAVED_RUNS to existing saved run directories before evaluating.")


,complete_images,incomplete_images,stable_images,changed_images,coverage,stable_fraction,stable_fraction_complete,valid_pairs,agreeing_pairs,pairwise_agreement
noise,50.0,146.0,50.0,0.0,0.255102,0.255102,1.0,150.0,150.0,1.0
severity,50.0,146.0,50.0,0.0,0.255102,0.255102,1.0,150.0,150.0,1.0
route,50.0,146.0,50.0,0.0,0.255102,0.255102,1.0,150.0,150.0,1.0
joint,50.0,146.0,50.0,0.0,0.255102,0.255102,1.0,150.0,150.0,1.0
